[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke06-klinisk-praksis/03_validering_generalisering_og_subgrupper.ipynb)

# 🔬 Validering, generalisering og subgrupper

I de to foregående notebookene så vi først hvordan en modell kan anslå risiko, og deretter hvordan risiko kan kobles til terskler og beslutninger. Denne notebooken stiller neste klinisk viktige spørsmål: Fungerer modellen fortsatt når den møter andre pasienter, andre arbeidsprosesser eller andre måleforhold enn dem den ble utviklet på?

## Læringsmål
- forklare forskjellen på intern og ekstern validering
- forstå hva generalisering betyr for kliniske modeller
- undersøke om en modell fungerer ulikt for ulike subgrupper
- reflektere over hvorfor distribusjonsskifte er viktig i medisinsk AI og klinisk implementering

### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab

In [ ]:
import sys, subprocess, os
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokalt miljø")

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print("✅ Miljø klart")

## Intern validering er ikke nok

En modell kan se god ut på testdata fra samme miljø som treningsdataene, men likevel fungere dårlig når den tas i bruk et annet sted. Dette er en klassisk utfordring i klinisk praksis: Modellen er utviklet under bestemte forhold, men skal ofte brukes i en virkelighet som er mer variert.

I klinisk praksis må vi derfor spørre:

- fungerer modellen på andre pasientgrupper enn dem den ble utviklet på?
- fungerer den med andre arbeidsflyter, instrumenter eller måleforhold?
- holder ytelsen seg når sykdomsbilde, henvisningsmønster eller datakvalitet endrer seg over tid?

Validering handler derfor ikke bare om å kontrollere modellen, men om å undersøke om den faktisk er trygg og relevant i den settingen der den skal brukes.

In [ ]:
# Syntetisk eksempel på distribusjonsskifte mellom treningsmiljø og nytt miljø

train_age = np.random.normal(55, 14, 1000)
train_age = train_age[(train_age > 20) & (train_age < 90)]

external_age = np.random.normal(73, 9, 500)
external_age = external_age[(external_age > 45) & (external_age < 95)]

plt.hist(train_age, bins=30, alpha=0.6, label='Treningsmiljø', density=True, edgecolor='black')
plt.hist(external_age, bins=30, alpha=0.6, label='Nytt miljø', density=True, edgecolor='black')
plt.axvline(train_age.mean(), linestyle='--', color='steelblue')
plt.axvline(external_age.mean(), linestyle='--', color='indianred')
plt.xlabel('Alder')
plt.ylabel('Tetthet')
plt.title('Generalisering blir vanskeligere når pasientgrunnlaget endrer seg')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Gjennomsnittlig alder i treningsmiljø: {train_age.mean():.1f} år")
print(f"Gjennomsnittlig alder i nytt miljø: {external_age.mean():.1f} år")

In [ ]:
# Syntetisk eksempel på forskjellig ytelse i subgrupper

grupper = ['Yngre pasienter', 'Eldre pasienter']
sensitivitet = [0.89, 0.72]
spesifisitet = [0.86, 0.81]

x = np.arange(len(grupper))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, sensitivitet, width, label='Sensitivitet', color='#2ecc71', edgecolor='black')
ax.bar(x + width/2, spesifisitet, width, label='Spesifisitet', color='#3498db', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(grupper)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Ytelse')
ax.set_title('Subgruppeanalyse: samme modell, ulik klinisk konsekvens')
ax.legend()
plt.tight_layout()
plt.show()

## Klinisk tolkning

Når en modell presterer ulikt i subgrupper, er det ikke bare et metodisk problem. Det kan få direkte kliniske konsekvenser. En modell som fungerer godt i gjennomsnitt, kan likevel være utilstrekkelig for bestemte pasientgrupper eller i bestemte deler av tjenesten.

Det kan bety:

- senere diagnose for noen pasienter
- ulik tilgang til behandling eller videre utredning
- at klinikere mister tillit til systemet fordi ytelsen virker ustabil eller uforutsigbar

Dette er også en viktig bro til uke 08, der vi ser på rettferdighet, ansvar og trustworthy AI. Men allerede her er poenget praktisk: Før en modell tas i bruk, må vi vite hvem den fungerer for, og hvem den eventuelt fungerer dårligere for.

### Refleksjon
- Hvorfor kan en modell fungere godt ett sted og dårlig et annet sted, selv om den ser god ut i utviklingsmiljøet?
- Hvilke subgrupper ville du undersøkt først i et ekte klinisk datasett, og hvorfor akkurat disse?
- Hvilke tegn på distribusjonsskifte ville du fulgt med på etter at modellen er tatt i bruk?
- Når bør svak subgruppeytelse føre til at en modell ikke tas i bruk, eller bare brukes med tydelige begrensninger?
- Hvordan bygger denne notebooken videre på notebook 01 og 02: Hvorfor er det ikke nok at modellen er forståelig og har en valgt terskel dersom den ikke generaliserer godt?